In [1]:
pip install conllu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
from collections import defaultdict
import requests
from conllu import parse
from sklearn.metrics import classification_report, accuracy_score

#--------------------
# Download & Load
#--------------------
def load_ud_ewt(split = "train"):
    urls = {
        "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu",
        "dev" : "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu"
    }
    response = requests.get(urls[split])
    sentences = []
    #2. Extract words and UPOS tags
    for tokenlist in parse(response.text):
        sent = [(token["form"], token["upos"]) for token in tokenlist if isinstance(token["id"], int)]
        if sent:
            sentences.append(sent)
    return sentences

#------------------------
# Transition and Emission Probabilities
#------------------------
class HMMPOSTagger:
    def __init__(self):
        self.tags = set()
        self.vocab = set()
        self.transition_counts = defaultdict(lambda: defaultdict(int))
        self.emission_counts = defaultdict(lambda: defaultdict(int))
        self.tag_counts = defaultdict(int)
        self.start_tag = "<START>"

    def fit(self, train_data):
        for sentence in train_data:
            prev_tag = self.start_tag
            for word, tag in sentence:
                self.tags.add(tag)
                self.vocab.add(word)
                self.transition_counts[prev_tag][tag] += 1
                self.emission_counts[tag][word] += 1
                prev_tag = tag
        self.tags = sorted(list(self.tags))
        self.vocab_size = len(self.vocab)
        self.num_tags = len(self.tags)

    def _log_trans_prob(self, prev_tag, curr_tag, alpha = 1.0):
        total_transitions = sum(self.transition_counts[prev_tag].values())
        count = self.transition_counts[prev_tag][curr_tag]
        return math.log((count + alpha) / (total_transitions + alpha * self.num_tags))

    def _log_emiss_prob(self, tag, word, alpha = 1e-4):
        total_emissions = self.tag_counts[tag]
        count = self.emission_counts[tag][word]
        return math.log((count + alpha) / (total_emissions + alpha * (self.vocab_size + 1)))

    #--------------------
    # Viterbi Algorithm Implementation
    #--------------------
    def predict(self, words):
        T = len(words)
        if T == 0:
            return []
        viterbi = [{}]
        backpointer = [{}]
        for tag in self.tags:
            log_trans = self._log_trans_prob(self.start_tag, tag)
            log_emiss = self._log_emiss_prob(tag, words[0])
            viterbi[0][tag] = log_trans + log_emiss
            backpointer[0][tag] = None

        for t in range(1, T):
            viterbi.append({})
            backpointer.append({})
            for curr_tag in self.tags:
                log_emiss = self._log_emiss_prob(curr_tag, words[t])
                max_log_prob, best_prev_tag = max(
                    (viterbi[t-1][prev_tag] + self._log_trans_prob(prev_tag, curr_tag) + log_emiss, prev_tag)
                    for prev_tag in self.tags
                )
                viterbi[t][curr_tag] = max_log_prob
                backpointer[t][curr_tag] = best_prev_tag

        best_last_tag = max(viterbi[T - 1], key = viterbi[T - 1].get)
        best_path = [best_last_tag]

        for t in range(T - 1, 0, -1):
            best_last_tag = backpointer[t][best_last_tag]
            best_path.insert(0, best_last_tag)

        return best_path

#--------------------
# Execution & Evaluation
#--------------------
print("Loading UD English-EWT datasets....")
train_sentences = load_ud_ewt("train")
dev_sentences = load_ud_ewt("dev")

print(f"Training on {len(train_sentences)} sentences....")
tagger = HMMPOSTagger()
tagger.fit(train_sentences)

#Predict arbitrary input sentence
sample_sentence = "The quick brown fox jumps over the lazy dog.".split()
sample_predictions = tagger.predict(sample_sentence)
print("\nSample Sentence predictions:")
for word, tag in zip(sample_sentence, sample_predictions):
    print(f"{word:<10} -> {tag}")

#Predicted vs Actual tags
print("\nEvaluating on Dev Set...")
y_true = []
y_pred = []
for sent in dev_sentences:
    words = [w for w, _ in sent]
    actual_tags = [t for _, t in sent]
    predicted_tags = tagger.predict(words)
    y_true.extend(actual_tags)
    y_pred.extend(predicted_tags)
print(f"\nOverall Token Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%\n")
print(classification_report(y_true, y_pred, zero_division = 0))

Loading UD English-EWT datasets....
Training on 12544 sentences....

Sample Sentence predictions:
The        -> DET
quick      -> ADJ
brown      -> NOUN
fox        -> AUX
jumps      -> VERB
over       -> ADP
the        -> DET
lazy       -> ADJ
dog.       -> NOUN

Evaluating on Dev Set...

Overall Token Accuracy: 88.10%

              precision    recall  f1-score   support

         ADJ       0.88      0.83      0.86      1865
         ADP       0.86      0.97      0.91      2038
         ADV       0.88      0.80      0.84      1232
         AUX       0.88      0.95      0.91      1567
       CCONJ       0.99      0.98      0.99       780
         DET       0.87      0.97      0.92      1900
        INTJ       0.99      0.57      0.73       115
        NOUN       0.83      0.91      0.86      4210
         NUM       0.98      0.66      0.79       383
        PART       0.94      0.88      0.91       647
        PRON       0.87      0.98      0.92      2225
       PROPN       0.87      